# Machine Learning Exercise 3
by Thanh Tran, Gero Brunke, Robin-Marcel Hanne

## Task 1: Introduction

In [13]:
import torch

print(f"CUDA Available: {torch.cuda.is_available()}")
print(f"MPS Available: {torch.backends.mps.is_available()}")  # For Apple Silicon M-series
print(f"Device Count: {torch.cuda.device_count()}")
if torch.cuda.is_available():
    print(f"Current Device Name: {torch.cuda.get_device_name(0)}")


CUDA Available: False
MPS Available: False
Device Count: 0


In [10]:
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.optim as optim
from torch.utils.data import DataLoader

import torchvision.datasets as datasets
import torchvision.transforms as transforms

from torch.nn.functional import conv2d, max_pool2d, cross_entropy

plt.rc("figure", dpi=100)

batch_size = 100

# transform images into normalized tensors
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.5,), std=(0.5,))
])

train_dataset = datasets.MNIST(
    "./",
    download=True,
    train=True,
    transform=transform,
)

test_dataset = datasets.MNIST(
    "./",
    download=True,
    train=False,
    transform=transform,
)

train_dataloader = DataLoader(
    dataset=train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=1,
    pin_memory=True,
)

test_dataloader = DataLoader(
    dataset=test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=1,
    pin_memory=True,
)

def init_weights(shape):
    # Kaiming He initialization (a good initialization is important)
    # https://arxiv.org/abs/1502.01852
    std = np.sqrt(2. / shape[0])
    w = torch.randn(size=shape) * std
    w.requires_grad = True
    return w


def rectify(x):
    # Rectified Linear Unit (ReLU)
    return torch.max(torch.zeros_like(x), x)


class RMSprop(optim.Optimizer):
    """
    This is a reduced version of the PyTorch internal RMSprop optimizer
    It serves here as an example
    """
    def __init__(self, params, lr=1e-3, alpha=0.5, eps=1e-8):
        defaults = dict(lr=lr, alpha=alpha, eps=eps)
        super(RMSprop, self).__init__(params, defaults)

    def step(self):
        for group in self.param_groups:
            for p in group['params']:
                grad = p.grad.data
                state = self.state[p]

                # state initialization
                if len(state) == 0:
                    state['square_avg'] = torch.zeros_like(p.data)

                square_avg = state['square_avg']
                alpha = group['alpha']

                # update running averages
                square_avg.mul_(alpha).addcmul_(grad, grad, value=1 - alpha)
                avg = square_avg.sqrt().add_(group['eps'])

                # gradient update
                p.data.addcdiv_(grad, avg, value=-group['lr'])


# define the neural network
def model(x, w_h, w_h2, w_o):
    h = rectify(x @ w_h)
    h2 = rectify(h @ w_h2)
    pre_softmax = h2 @ w_o
    return pre_softmax


# initialize weights

# input shape is (B, 784)
w_h = init_weights((784, 625))
# hidden layer with 625 neurons
w_h2 = init_weights((625, 625))
# hidden layer with 625 neurons
w_o = init_weights((625, 10))
# output shape is (B, 10)

optimizer = RMSprop(params=[w_h, w_h2, w_o])


n_epochs = 100

train_loss = []
test_loss = []

# put this into a training loop over 100 epochs
for epoch in range(n_epochs + 1):
    train_loss_this_epoch = []
    for idx, batch in enumerate(train_dataloader):
        x, y = batch

        # our model requires flattened input
        x = x.reshape(batch_size, 784)
        # feed input through model
        noise_py_x = model(x, w_h, w_h2, w_o)

        # reset the gradient
        optimizer.zero_grad()

        # the cross-entropy loss function already contains the softmax
        loss = cross_entropy(noise_py_x, y, reduction="mean")

        train_loss_this_epoch.append(float(loss))

        # compute the gradient
        loss.backward()
        # update weights
        optimizer.step()

    train_loss.append(np.mean(train_loss_this_epoch))

    # test periodically
    if epoch % 10 == 0:
        print(f"Epoch: {epoch}")
        print(f"Mean Train Loss: {train_loss[-1]:.2e}")
        test_loss_this_epoch = []

        # no need to compute gradients for validation
        with torch.no_grad():
            for idx, batch in enumerate(test_dataloader):
                x, y = batch
                x = x.reshape(batch_size, 784)
                noise_py_x = model(x, w_h, w_h2, w_o)

                loss = cross_entropy(noise_py_x, y, reduction="mean")
                test_loss_this_epoch.append(float(loss))

        test_loss.append(np.mean(test_loss_this_epoch))

        print(f"Mean Test Loss:  {test_loss[-1]:.2e}")

plt.plot(np.arange(n_epochs + 1), train_loss, label="Train")
plt.plot(np.arange(1, n_epochs + 2, 10), test_loss, label="Test")
plt.title("Train and Test Loss over Training")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()

100.0%
100.0%
100.0%
100.0%
g:\Users\smood\anaconda3\envs\ml_homework\Lib\site-packages\torch\utils\data\dataloader.py:1095: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
C:\Users\smood\AppData\Local\Temp\ipykernel_33608\2286586689.py:140: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\autograd\generated\python_variable_methods.cpp:839.)
  train_loss_this_epoch.append(float(loss))


Epoch: 0
Mean Train Loss: 3.91e-01
Mean Test Loss:  1.50e-01
Epoch: 10
Mean Train Loss: 1.54e-01
Mean Test Loss:  2.35e-01
Epoch: 20
Mean Train Loss: 1.10e-01
Mean Test Loss:  3.80e-01
Epoch: 30
Mean Train Loss: 5.79e-02
Mean Test Loss:  6.92e-01
Epoch: 40
Mean Train Loss: 5.49e-02
Mean Test Loss:  5.81e-01
Epoch: 50
Mean Train Loss: 4.23e-02
Mean Test Loss:  8.87e-01
Epoch: 60
Mean Train Loss: 3.27e-02
Mean Test Loss:  8.15e-01


KeyboardInterrupt: 

## Task 2: Dropout

In [ ]:
def dropout(X, p_drop=0.5):
    if p_drop <= 0 or p_drop >= 1:
        return X

    mask = torch.bernoulli(torch.full_like(X, p_drop))

    return (X * (1 - mask)) / (1 - p_drop)

In [ ]:
def dropout_model(X, w_h, w_h2, w_o, p_drop_input=0.2, p_drop_hidden=0.5):
    X = dropout(X, p_drop_input)
    h = rectify(X @ w_h)
    h = dropout(h, p_drop_hidden)
    h2 = rectify(h @ w_h2)
    h2 = dropout(h2, p_drop_hidden)
    pre_softmax = h2 @ w_o
    return pre_softmax

In [ ]:
# initialize weights

# input shape is (B, 784)
w_h = init_weights((784, 625))
# hidden layer with 625 neurons
w_h2 = init_weights((625, 625))
# hidden layer with 625 neurons
w_o = init_weights((625, 10))
# output shape is (B, 10)

optimizer = RMSprop(params=[w_h, w_h2, w_o])


n_epochs = 100

train_loss = []
test_loss = []

# put this into a training loop over 100 epochs
for epoch in range(n_epochs + 1):
    train_loss_this_epoch = []
    for idx, batch in enumerate(train_dataloader):
        x, y = batch

        # our model requires flattened input
        x = x.reshape(batch_size, 784)
        # feed input through model
        noise_py_x = dropout_model(x, w_h, w_h2, w_o)

        # reset the gradient
        optimizer.zero_grad()

        # the cross-entropy loss function already contains the softmax
        loss = cross_entropy(noise_py_x, y, reduction="mean")

        train_loss_this_epoch.append(float(loss))

        # compute the gradient
        loss.backward()
        # update weights
        optimizer.step()

    train_loss.append(np.mean(train_loss_this_epoch))

    # test periodically
    if epoch % 10 == 0:
        print(f"Epoch: {epoch}")
        print(f"Mean Train Loss: {train_loss[-1]:.2e}")
        test_loss_this_epoch = []

        # no need to compute gradients for validation
        with torch.no_grad():
            for idx, batch in enumerate(test_dataloader):
                x, y = batch
                x = x.reshape(batch_size, 784)
                noise_py_x = model(x, w_h, w_h2, w_o)

                loss = cross_entropy(noise_py_x, y, reduction="mean")
                test_loss_this_epoch.append(float(loss))

        test_loss.append(np.mean(test_loss_this_epoch))

        print(f"Mean Test Loss:  {test_loss[-1]:.2e}")

plt.plot(np.arange(n_epochs + 1), train_loss, label="Train")
plt.plot(np.arange(1, n_epochs + 2, 10), test_loss, label="Test")
plt.title("Train and Test Loss over Training")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()

### How the dropout method works and how it reduces overfitting
Dropout works by randomly setting a fraction of the activations in a layer to zero during training. Because neurons cannot rely on the presence of specific other neurons, the model is forced to learn more robust and redundant features instead of memorizing patterns tied to particular activations. This reduces overfitting by preventing co-adaptation of neurons.

### Why we need a different model configuration for evaluating the test loss
We need a different model configuration for evaluating the test loss because dropout introduces randomness during training by removing activations, but at test time we want deterministic and consistent predictions using the full network. If dropout were still active during evaluation, the output would vary randomly for the same input, making the test loss unreliable. Therefore, dropout is disabled during testing so that all neurons contribute and the learned weights are used at full capacity.

### Comparison of the errors from Section 1
In the first model without dropout, the training loss decreases strongly and continuously down to very small values, while the test loss initially decreases but then steadily increases after around epoch 10–20. The model overfits heavily.

In the dropout model, the training loss stays significantly higher and does not decrease nearly as much, which is expected because dropout makes the optimization task harder by randomly removing parts of the network during training. However, the important difference is that the test loss is lower and much more stable across epochs.

Compared to the first model in Section 1, the dropout model clearly reduces overfitting. While the baseline model achieves a lower training loss, it ends up with a much worse final test loss, whereas the dropout model maintains a substantially lower and more stable test loss. This shows that dropout improves generalization at the cost of higher training error.

## Task 3: Parametric Relu

Implementing the PreLU function

$x_i \rightarrow
    \begin{cases}
    x_i      & x_i > 0\\
    a_i x_i  & x_i \leq 0
    \end{cases}
$

In [ ]:
def PReLU ( X, a ):
    """
    Parametric rectified Linear Unit
    Applies X if X >Y 0 and x * X if X <= 0
    """
    return torch.where( X > 0, X, a * X )

And updated model architecutre with PRelu instead of recify - this will rely on task 1

In [ ]:
def PReLU_model(x, w_h, w_h2, w_o, a_h, a_h2):
    # 1st layer
    h = PReLU(x @ w_h, a_h)

    # 2nd layer (hidden)
    h2 = PReLU(h @ w_h2, a_h2)

    pre_smx = h2 @ w_o
    return pre_smx

Training requires grad so new torches

In [ ]:
# initialize weights

# input shape is (B, 784)
w_h = init_weights((784, 625))
# hidden layer with 625 neurons
w_h2 = init_weights((625, 625))
# hidden layer with 625 neurons
w_o = init_weights((625, 10))
# output shape is (B, 10)

a_h = torch.full((625,), 0.25, requires_grad=True)
a_h2 = torch.full((625,), 0.25, requires_grad=True)

# including a_h and a_h2

optimizer = RMSprop(params=[w_h, w_h2, w_o, a_h, a_h2])

n_epochs = 100

train_loss = []
test_loss = []

# put this into a training loop over 100 epochs
for epoch in range(n_epochs + 1):
    train_loss_this_epoch = []
    for idx, batch in enumerate(train_dataloader):
        x, y = batch

        # our model requires flattened input
        x = x.reshape(batch_size, 784)
        # feed input through model
        noise_py_x = PReLU_model(x, w_h, w_h2, w_o, a_h, a_h2)

        # reset the gradient
        optimizer.zero_grad()

        # the cross-entropy loss function already contains the softmax
        loss = cross_entropy(noise_py_x, y, reduction="mean")

        train_loss_this_epoch.append(float(loss))

        # compute the gradient
        loss.backward()
        # update weights
        optimizer.step()

    train_loss.append(np.mean(train_loss_this_epoch))

    # test periodically
    if epoch % 10 == 0:
        print(f"Epoch: {epoch}")
        print(f"Mean Train Loss: {train_loss[-1]:.2e}")
        test_loss_this_epoch = []

        # no need to compute gradients for validation
        with torch.no_grad():
            for idx, batch in enumerate(test_dataloader):
                x, y = batch
                x = x.reshape(batch_size, 784)
                noise_py_x = PReLU_model(x, w_h, w_h2, w_o, a_h, a_h2)

                loss = cross_entropy(noise_py_x, y, reduction="mean")
                test_loss_this_epoch.append(float(loss))

        test_loss.append(np.mean(test_loss_this_epoch))

        print(f"Mean Test Loss:  {test_loss[-1]:.2e}")

plt.plot(np.arange(n_epochs + 1), train_loss, label="Train")
plt.plot(np.arange(1, n_epochs + 2, 10), test_loss, label="Test")
plt.title("Train and Test Loss over Training")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()

# Comparing
 - **Compared to task 1:** The PReLU converges slightly faster and achieves a lower training loss. Because the negative inputs do not lead to a 0 less nuerons die leading to a higher learning capacity.

 - **Compared to task 2:** The PReLU doesn't regularize the network like the Dropout does. It might overfit despite being a lower training error.